# 03a — Assemble the physicochemical feature matrix (TEM-1 / Firnberg)

Builds the **physicochemical modeling table** the traditional-ML benchmark (notebook `04`) trains on.
This is the no-language-model, no-identity feature representation: for each single missense variant we
describe the substitution purely by its **physical chemistry** — the change in hydrophobicity, charge,
side-chain volume, polarity, flexibility, and the standard multidimensional property scales — with no
amino-acid or position identity given to the model at all.

Companion to `00a`-style dataset construction: it reads the AA-identity modeling table (built earlier)
for the variant list and labels, and emits a parallel table with identical rows/labels but a
physicochemical feature block. Notebook `03` (EDA) and `04` (benchmark) read the output of this step
from disk.

**Design goal.** Cover the physical-property space as completely as possible, so that when the benchmark
plateaus we can say the hand-built-chemistry option was *exhausted*, not under-specified.

In [1]:
# self-contained: resolve project root via .projectroot, read from disk
import sys
from pathlib import Path
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'.projectroot').exists())
sys.path.insert(0, str(root))
from paths import *   # BASE_DIR, PROCESSED, ...

import numpy as np, pandas as pd, json, re
import peptides
import peptides.tables as T

AA = list("ACDEFGHIKLMNPQRSTVWY")
OUTDIR = PROCESSED / "physicochemical"; OUTDIR.mkdir(parents=True, exist_ok=True)
print("root:", root)

root: /Users/kdave2/Beta-Lactamase Mutagenesis/1 - ML


## 1. Load the variant list and labels

Read the AA-identity modeling table — the single source of truth for which variants exist and their
labels. We reuse its rows verbatim (CLAUDE.md: reuse over rebuild) and only swap the feature block.

In [2]:
src = PROCESSED / "traditional_ml_aa_identity" / "modeling_dataset.parquet"
md = pd.read_parquet(src).sort_values("seq_id").reset_index(drop=True)   # deterministic order
assert {"seq_id","wt_aa","mut_aa","position","DMS_score","DMS_score_bin"} <= set(md.columns)
assert md.wt_aa.isin(AA).all() and md.mut_aa.isin(AA).all()
print("variants:", len(md), "| positions:", md.position.nunique(),
      "| class balance:", md.DMS_score_bin.value_counts().to_dict())

variants: 4783 | positions: 263 | class balance: {1: 2397, 0: 2386}


## 2. Per-residue property tables

Two provenance classes, kept explicit:

- **From the `peptides` library** (queried, not typed): Kyte-Doolittle and Eisenberg hydrophobicity
  scales, per-residue molecular weight, and the full multidimensional QSAR descriptor set
  (`descriptors()` — 102 dims spanning Z-scales, VHSE, T-scales, Kidera, FASGAI, ProtFP, BLOSUM
  indices, Atchley, Cruciani, MS-WHIM, ST, and others).
- **Standard literature constants, hand-entered as literal tables**: Zamyatnin side-chain volume,
  Grantham polarity, Chou-Fasman α/β propensity, Bhaskaran-Ponnuswamy flexibility, side-chain H-bond
  donor/acceptor counts, formal charge at pH 7 (His as +0.1), and standard free-amino-acid pI. These
  are cross-validated against the project's existing engineered `dphys_` columns in step 4.

In [3]:
# --- from peptides (library lookups) ---
KD    = T.HYDROPHOBICITY["KyteDoolittle"]
EISEN = T.HYDROPHOBICITY["Eisenberg"]
MW    = {a: peptides.Peptide(a).molecular_weight() for a in AA}

# --- standard literature constants (hand-entered; validated in step 4) ---
# Zamyatnin side-chain volume (A^3)
VOL = dict(A=88.6,R=173.4,N=114.1,D=111.1,C=108.5,Q=143.8,E=138.4,G=60.1,H=153.2,
           I=166.7,L=166.7,K=168.6,M=162.9,F=189.9,P=112.7,S=89.0,T=116.1,W=227.8,Y=193.6,V=140.0)
# formal charge at pH 7 (His partial +0.1 -- matches project dphys_charge)
CHG = {a:0.0 for a in AA}; CHG.update(D=-1.0,E=-1.0,K=1.0,R=1.0,H=0.1)
POLG = dict(A=8.1,R=10.5,N=11.6,D=13.0,C=5.5,Q=10.5,E=12.3,G=9.0,H=10.4,I=5.2,
            L=4.9,K=11.3,M=5.7,F=5.2,P=8.0,S=9.2,T=8.6,W=5.4,Y=6.2,V=5.9)          # Grantham polarity
CFA = dict(A=1.42,R=0.98,N=0.67,D=1.01,C=0.70,Q=1.11,E=1.51,G=0.57,H=1.00,I=1.08,
           L=1.21,K=1.16,M=1.45,F=1.13,P=0.57,S=0.77,T=0.83,W=1.08,Y=0.69,V=1.06)   # Chou-Fasman alpha
CFB = dict(A=0.83,R=0.93,N=0.89,D=0.54,C=1.19,Q=1.10,E=0.37,G=0.75,H=0.87,I=1.60,
           L=1.30,K=0.74,M=1.05,F=1.38,P=0.55,S=0.75,T=1.19,W=1.37,Y=1.47,V=1.70)   # Chou-Fasman beta
FLEX = dict(A=0.360,R=0.529,N=0.463,D=0.511,C=0.346,Q=0.493,E=0.497,G=0.544,H=0.323,
            I=0.462,L=0.365,K=0.466,M=0.295,F=0.314,P=0.509,S=0.507,T=0.444,W=0.305,Y=0.420,V=0.386)  # B-P flexibility
HBD = dict(A=0,R=3,N=1,D=0,C=1,Q=1,E=0,G=0,H=1,I=0,L=0,K=1,M=0,F=0,P=0,S=1,T=1,W=1,Y=1,V=0)  # sidechain H-bond donors
HBA = dict(A=0,R=0,N=1,D=2,C=0,Q=1,E=2,G=0,H=1,I=0,L=0,K=0,M=0,F=0,P=0,S=1,T=1,W=0,Y=1,V=0)  # acceptors
PI  = dict(A=6.00,R=10.76,N=5.41,D=2.77,C=5.07,E=3.22,Q=5.65,G=5.97,H=7.59,I=6.02,
           L=5.98,K=9.74,M=5.74,F=5.48,P=6.30,S=5.68,T=5.60,W=5.89,Y=5.66,V=5.96)   # free-AA pI
AROM = {a:(1 if a in "FWYH" else 0) for a in AA}
ALIP = {a:(1 if a in "AVLI" else 0) for a in AA}

SCALAR_TABLES = dict(hydropathy_kd=KD, hydrophobicity_eisenberg=EISEN, volume=VOL, charge=CHG,
    polarity_grantham=POLG, pI=PI, mw=MW, cf_alpha=CFA, cf_beta=CFB, flexibility=FLEX,
    hbond_donors=HBD, hbond_acceptors=HBA, aromatic=AROM, aliphatic=ALIP)

# 102-dim QSAR descriptor vector per residue, cached
DESC = {a: peptides.Peptide(a).descriptors() for a in AA}
desc_names = list(DESC["A"].keys())
print(len(SCALAR_TABLES), "interpretable scalars +", len(desc_names), "QSAR descriptor dims per residue")

14 interpretable scalars + 102 QSAR descriptor dims per residue


## 3. Build the feature block

Each interpretable scalar and each QSAR descriptor dim is expanded to three columns — **wild-type value,
mutant value, and their difference (mut − wt)** — so the model sees both the absolute chemistry and the
change. On top of that, **categorical switch flags** encode chemistry transitions a smooth delta can
miss (proline/glycine gain or loss, charge-sign flip, hydrophobic↔hydrophilic switch, aromatic
gain/loss, large size change).

In [4]:
def build_row(wt, mut):
    f = {}
    for name, tab in SCALAR_TABLES.items():
        wv, mv = tab[wt], tab[mut]
        f[f"{name}_wt"] = wv; f[f"{name}_mut"] = mv; f[f"{name}_delta"] = mv - wv
    dw, dm = DESC[wt], DESC[mut]
    for k in desc_names:
        f[f"desc_{k}_wt"] = dw[k]; f[f"desc_{k}_mut"] = dm[k]; f[f"desc_{k}_delta"] = dm[k] - dw[k]
    # switch flags -- transitions a scalar delta alone can't represent
    f["flag_pro_gain"]=int(mut=="P" and wt!="P"); f["flag_pro_loss"]=int(wt=="P" and mut!="P")
    f["flag_gly_gain"]=int(mut=="G" and wt!="G"); f["flag_gly_loss"]=int(wt=="G" and mut!="G")
    f["flag_cys_gain"]=int(mut=="C" and wt!="C"); f["flag_cys_loss"]=int(wt=="C" and mut!="C")
    f["flag_charge_flip"]=int(np.sign(CHG[wt])!=np.sign(CHG[mut]) and (CHG[wt]!=0 or CHG[mut]!=0))
    f["flag_charge_gain"]=int(CHG[wt]==0 and CHG[mut]!=0); f["flag_charge_loss"]=int(CHG[wt]!=0 and CHG[mut]==0)
    f["flag_hydro_switch"]=int(np.sign(KD[wt])!=np.sign(KD[mut]))
    f["flag_arom_gain"]=int(AROM[mut]==1 and AROM[wt]==0); f["flag_arom_loss"]=int(AROM[wt]==1 and AROM[mut]==0)
    f["flag_size_big_to_small"]=int(VOL[wt]-VOL[mut]>40); f["flag_size_small_to_big"]=int(VOL[mut]-VOL[wt]>40)
    return f

feat = pd.DataFrame([build_row(r.wt_aa, r.mut_aa) for r in md.itertuples()])
X_full = pd.concat([md[["seq_id","wt_aa","mut_aa","position","DMS_score","DMS_score_bin"]].reset_index(drop=True),
                    feat], axis=1)
feat_cols = list(feat.columns)
groups = {"interpretable (wt/mut/delta)": sum(1 for c in feat_cols if any(c.startswith(s+"_") for s in SCALAR_TABLES)),
          "QSAR descriptors (wt/mut/delta)": sum(1 for c in feat_cols if c.startswith("desc_")),
          "categorical flags": sum(1 for c in feat_cols if c.startswith("flag_"))}
print("matrix:", X_full.shape, "|", groups)

matrix: (4783, 368) | {'interpretable (wt/mut/delta)': 42, 'QSAR descriptors (wt/mut/delta)': 306, 'categorical flags': 14}


## 4. Validate at the boundary (CLAUDE.md)

Nothing downstream trusts this table until it passes: (i) it aligns row-for-row with the AA-identity
table (same `seq_id`s, identical labels and DMS scores), (ii) no NaN/inf, (iii) the interpretable
deltas reproduce the project's existing engineered `dphys_` columns, confirming the property tables are
correct, and (iv) no feature is a label proxy (leakage guard). Assert, don't warn.

In [5]:
# (i) alignment vs the AA-identity table
ref = md.set_index("seq_id"); cur = X_full.set_index("seq_id")
assert set(cur.index) == set(ref.index), "seq_id set mismatch"
ref = ref.loc[cur.index]
assert (cur.DMS_score_bin.values == ref.DMS_score_bin.values).all(), "label mismatch"
assert np.allclose(cur.DMS_score.values, ref.DMS_score.values), "DMS_score mismatch"
assert (cur.wt_aa.values==ref.wt_aa.values).all() and (cur.mut_aa.values==ref.mut_aa.values).all()

# (ii) no NaN / inf
y = X_full.DMS_score_bin.values.astype(int)
Xv = X_full[feat_cols].values.astype(float)
assert not np.isnan(Xv).any() and not np.isinf(Xv).any(), "NaN/inf in features"

# (iii) interpretable deltas reproduce the project's engineered dphys_ columns
ef_path = BASE_DIR.parent / "engineered_features.parquet"
if ef_path.exists():
    ef = pd.read_parquet(ef_path).set_index("mutant").loc[X_full.seq_id.values]
    checks = {"dphys_hydropathy":KD,"dphys_volume":VOL,"dphys_charge":CHG,"dphys_MW":MW,
              "dphys_pI":PI,"dphys_cf_alpha":CFA,"dphys_cf_beta":CFB}
    for col, tab in checks.items():
        mine = np.array([tab[m]-tab[w] for w,m in zip(X_full.wt_aa, X_full.mut_aa)])
        d = np.abs(mine - ef[col].values).max()
        assert d < 0.05, f"{col} delta disagrees with engineered_features ({d:.3f})"
    print("interpretable deltas reproduce engineered dphys_ columns (max err < 0.05)")
else:
    print("engineered_features.parquet not found -- skipping dphys_ cross-check")

# (iv) leakage guard: no target-derived column, nothing perfectly tracks y
assert not any(any(t in c.lower() for t in ["dms","score","bin","label"]) for c in feat_cols)
active = np.where(Xv.std(0) > 0)[0]
mx = max(abs(np.corrcoef(Xv[:,j], y)[0,1]) for j in active)
assert mx < 0.999, f"leakage: a feature tracks the label (r={mx:.3f})"
print(f"boundary checks passed | max |corr(feature,y)| = {mx:.3f} | constant cols = {len(feat_cols)-len(active)}")

interpretable deltas reproduce engineered dphys_ columns (max err < 0.05)
boundary checks passed | max |corr(feature,y)| = 0.220 | constant cols = 0


## 5. Save the modeling table and feature metadata

The table goes to `data/processed/physicochemical/`. Alongside it we write `feature_metadata.json`
(CLAUDE.md: bundle artifacts with what they need to be reused) — the feature column list, per-column
family / subfamily / kind (wt|mut|delta|flag), the interpretable-scalar names, the QSAR scale families,
and the exact provenance of every property table.

In [6]:
def fam(c):
    if c.startswith("flag_"): return "flag"
    if c.startswith("desc_"): return "qsar_scale"
    return "interpretable"
def subfam(c):
    if c.startswith("flag_"): return "flag"
    if c.startswith("desc_"): return re.sub(r"\d+$","", c[len("desc_"):].rsplit("_",1)[0])
    return c.rsplit("_",1)[0]
def kind(c):
    return "flag" if c.startswith("flag_") else c.rsplit("_",1)[1]

X_full.to_parquet(OUTDIR/"modeling_dataset.parquet", index=False)
meta = {"n_variants":int(len(X_full)), "n_features":len(feat_cols), "feature_columns":feat_cols,
        "id_columns":["seq_id","wt_aa","mut_aa","position"],
        "label_column":"DMS_score_bin", "score_column":"DMS_score",
        "groups":{c:fam(c) for c in feat_cols}, "subfamilies":{c:subfam(c) for c in feat_cols},
        "kind":{c:kind(c) for c in feat_cols},
        "interpretable_scalars":list(SCALAR_TABLES.keys()),
        "qsar_scale_families":sorted(set(subfam(c) for c in feat_cols if c.startswith("desc_"))),
        "source_tables":("QSAR scales + KyteDoolittle/Eisenberg hydrophobicity + per-residue MW from "
            "the peptides library (queried). Zamyatnin volume, Grantham polarity, Chou-Fasman alpha/beta, "
            "Bhaskaran-Ponnuswamy flexibility, sidechain H-bond donor/acceptor counts, formal charge "
            "(His +0.1), free-AA pI: standard literature constants hand-entered as literal tables. "
            "hydropathy/volume/charge/MW/pI/cf deltas cross-validated against engineered_features dphys_ "
            "columns to <0.05 abs error."),
        "provenance":"data/processed/traditional_ml_aa_identity/modeling_dataset.parquet (same seq_ids/labels)"}
json.dump(meta, open(OUTDIR/"feature_metadata.json","w"), indent=2)
print("wrote:", (OUTDIR/'modeling_dataset.parquet').relative_to(BASE_DIR))
print("wrote:", (OUTDIR/'feature_metadata.json').relative_to(BASE_DIR))
print("group counts:", {k:sum(1 for c in feat_cols if fam(c)==k) for k in ['interpretable','qsar_scale','flag']})

wrote: data/processed/physicochemical/modeling_dataset.parquet
wrote: data/processed/physicochemical/feature_metadata.json
group counts: {'interpretable': 42, 'qsar_scale': 306, 'flag': 14}
